# Preprocessing Pipeline — Option 4 Split

**New split strategy (do not overwrite existing `preprocessing.ipynb` output).**

| Set | Source | Windows |
|-----|--------|---------|
| Training | SC1 all normals + SC2 all normals + CC1 normals 80% | ~274k |
| Validation (early stop) | 10% carved from training | ~30k |
| Test Simple | CC1 normals 20% + CC1 anomalies | ~45k |
| Test Drift | CC2 all windows | ~77k |

**Why this split?**
- VAE trains on a large, diverse pool of normal behaviour (SC1 + SC2 + most of CC1)
- Test Simple evaluates in-distribution held-out CC1 (sudden faults: CPU spike, loss)
- Test Drift evaluates CC2 complex multi-fault / gradual degradation scenarios
- Outputs saved to `data/processed/windows_v4/` — no conflict with original outputs

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, os, collections
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

BASE        = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3\data'
IN_PATH     = os.path.join(BASE, 'processed', 'all_cases_labeled.csv')
OUT_DIR     = os.path.join(BASE, 'processed', 'windows_v4')
os.makedirs(OUT_DIR, exist_ok=True)

WINDOW_SIZE     = 30       # 30 timesteps x 15s = 7.5 minutes
STRIDE          = 1
GAP_THRESH      = 30       # seconds — time diff above this = gap boundary
PCA_VARIANCE    = 0.95     # keep components explaining 95% variance
CC1_TRAIN_RATIO = 0.80     # 80% CC1 normals -> train, 20% -> test_simple
VAL_RATIO       = 0.10     # 10% of assembled training -> validation (early stop)
RANDOM_SEED     = 42

FEATURE_COLS = [
    'container_cpu_usage_seconds_rate',
    'container_cpu_system_seconds_rate',
    'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes',
    'container_memory_working_set_bytes',
    'container_memory_rss',
    'container_memory_cache',
]
N_FEATURES = len(FEATURE_COLS)           # 7
FLAT_SIZE  = WINDOW_SIZE * N_FEATURES    # 210

print('Config ready.')
print(f'  Window       : {WINDOW_SIZE} steps  ({WINDOW_SIZE*15//60} min {WINDOW_SIZE*15%60} sec)')
print(f'  PCA variance : {PCA_VARIANCE*100:.0f}%')
print(f'  CC1 train    : {CC1_TRAIN_RATIO*100:.0f}%  /  CC1 test: {(1-CC1_TRAIN_RATIO)*100:.0f}%')
print(f'  Val ratio    : {VAL_RATIO*100:.0f}% of assembled training')
print(f'  Output dir   : {OUT_DIR}')

Config ready.
  Window       : 30 steps  (7 min 30 sec)
  PCA variance : 95%
  CC1 train    : 80%  /  CC1 test: 20%
  Val ratio    : 10% of assembled training
  Output dir   : c:\Users\jthar\Documents\Claude\Projects\module3\data\processed\windows_v4


---
## Step 1 — Load and Clean
Remove duplicate timestamps and mark gap boundaries per container.
This step is identical to the original preprocessing.

In [10]:
def clean_container(c_df):
    c = c_df.sort_values('timestamp').copy()
    c = c.drop_duplicates(subset='timestamp', keep='first').reset_index(drop=True)
    c['time_diff'] = c['timestamp'].diff().fillna(15)
    c['is_gap']    = c['time_diff'] > GAP_THRESH
    return c

print('Loading merged labeled file ...')
raw = pd.read_csv(IN_PATH, low_memory=False)
print(f'Loaded  : {len(raw):,} rows x {raw.shape[1]} columns')
print(f'Cases   : {raw["case"].unique().tolist()}')
print()

cleaned_parts = []
for case in ['single_case1', 'single_case2', 'complex_case1', 'complex_case2']:
    case_df = raw[raw['case'] == case]
    parts   = [clean_container(case_df[case_df['cmdb_id'] == cid])
               for cid in case_df['cmdb_id'].unique()]
    clean   = pd.concat(parts, ignore_index=True)
    removed = len(case_df) - len(clean)
    gaps    = int(clean['is_gap'].sum())
    cleaned_parts.append(clean)
    print(f'  {case}: {len(case_df):>7,} -> {len(clean):>7,} rows  '
          f'dups_removed={removed:>6,}  gap_boundaries={gaps}')

df = pd.concat(cleaned_parts, ignore_index=True)
print(f'\nTotal clean rows: {len(df):,}')

Loading merged labeled file ...
Loaded  : 533,230 rows x 12 columns
Cases   : ['single_case1', 'single_case2', 'complex_case1', 'complex_case2']

  single_case1:  74,196 ->  60,480 rows  dups_removed=13,716  gap_boundaries=0
  single_case2:  45,711 ->  38,853 rows  dups_removed= 6,858  gap_boundaries=0
  complex_case1: 318,418 -> 223,803 rows  dups_removed=94,615  gap_boundaries=54
  complex_case2:  94,905 ->  77,760 rows  dups_removed=17,145  gap_boundaries=0

Total clean rows: 400,896


---
## Step 2 — Sliding Window
Build rolling windows of size 30 per container.  
Windows crossing a gap boundary are skipped.  
A window is labelled anomaly (y=1) if **any** timestep within it is anomalous.

In [11]:
def build_windows(container_df, window_size, feature_cols):
    data   = container_df[feature_cols].values.astype(np.float32)
    labels = container_df['label'].values
    ftypes = container_df['failure_type'].values.astype(str)
    is_gap = container_df['is_gap'].values
    n      = len(data)
    X, y, ft = [], [], []
    for i in range(n - window_size):
        if is_gap[i : i + window_size].any():
            continue
        X.append(data[i : i + window_size])
        y.append(int(labels[i : i + window_size].any()))
        w_ft    = ftypes[i : i + window_size]
        anomaly = [v for v in w_ft if v not in ('nan', 'NaN', 'None', '')]
        ft.append(anomaly[0] if anomaly else np.nan)
    return (np.array(X, dtype=np.float32),
            np.array(y, dtype=np.int8),
            np.array(ft, dtype=object))

print(f'Building sliding windows (size={WINDOW_SIZE}, stride={STRIDE}) ...\n')
case_windows = {}
for case in ['single_case1', 'single_case2', 'complex_case1', 'complex_case2']:
    case_df              = df[df['case'] == case]
    X_list, y_list, ft_list = [], [], []
    for cid in sorted(case_df['cmdb_id'].unique()):
        c = case_df[case_df['cmdb_id'] == cid].sort_values('timestamp').reset_index(drop=True)
        Xc, yc, ftc = build_windows(c, WINDOW_SIZE, FEATURE_COLS)
        if len(Xc) > 0:
            X_list.append(Xc); y_list.append(yc); ft_list.append(ftc)
    X  = np.concatenate(X_list,  axis=0)
    y  = np.concatenate(y_list,  axis=0)
    ft = np.concatenate(ft_list, axis=0)
    case_windows[case] = {'X': X, 'y': y, 'ft': ft}
    print(f'  {case}:')
    print(f'    Total windows   : {len(X):>7,}')
    print(f'    Normal  (y=0)   : {(y==0).sum():>7,}')
    print(f'    Anomaly (y=1)   : {(y==1).sum():>7,}  ({y.mean()*100:.2f}%)')
    print()

Building sliding windows (size=30, stride=1) ...

  single_case1:
    Total windows   :  59,670
    Normal  (y=0)   :  59,467
    Anomaly (y=1)   :     203  (0.34%)

  single_case2:
    Total windows   :  38,043
    Normal  (y=0)   :  37,958
    Anomaly (y=1)   :      85  (0.22%)

  complex_case1:
    Total windows   : 221,495
    Normal  (y=0)   : 220,712
    Anomaly (y=1)   :     783  (0.35%)

  complex_case2:
    Total windows   :  76,950
    Normal  (y=0)   :  75,471
    Anomaly (y=1)   :   1,479  (1.92%)



---
## Step 3 — Memory Delta Features + Flatten

**Why memory deltas?**
- CPU metrics are already rates (`container_cpu_usage_seconds_rate`) — the "how fast" signal is built in
- Memory metrics are raw gauge values (bytes) — they need an explicit delta to expose slow growth trends (memory leaks)
- `delta[t] = memory[t] - memory[t-1]` directly says "how many bytes grew in the last 15 seconds"
- This signal can survive PCA compression better than two similar-looking raw values

**What we compute:**
- Original window: `(30, 7)` → flatten → `210` values
- Memory delta: `diff(memory columns, axis=time)` → `(29, 4)` → flatten → `116` values  
- Combined input to PCA: `210 + 116 = 326` values

In [12]:
# Memory column indices inside the 7-feature array
# [cpu_usage, cpu_system, cpu_user, mem_usage, mem_wset, mem_rss, mem_cache]
#       0          1          2         3          4         5         6
MEM_IDX = [3, 4, 5, 6]   # the 4 memory columns

print('Step 3a — Computing memory delta features ...')
print()
for case in case_windows:
    X = case_windows[case]['X']           # (N, 30, 7)
    X_mem   = X[:, :, MEM_IDX]           # (N, 30, 4)  — memory columns only
    X_delta = np.diff(X_mem, axis=1)     # (N, 29, 4)  — 29 step-by-step differences
    case_windows[case]['X_delta'] = X_delta
    print(f'  {case}: {X_mem.shape} -> diff -> {X_delta.shape}')

print()
print(f'  Delta meaning: X_delta[window, t] = memory[t+1] - memory[t]')
print(f'  Positive delta = memory growing  |  Negative = shrinking  |  ~0 = stable')
print()

print('Step 3b — Flattening and concatenating ...')
print()
for case in case_windows:
    X_raw   = case_windows[case]['X']         # (N, 30, 7)
    X_delta = case_windows[case]['X_delta']   # (N, 29, 4)

    X_flat_raw   = X_raw.reshape(len(X_raw), -1)       # (N, 210)
    X_flat_delta = X_delta.reshape(len(X_delta), -1)   # (N, 116)

    # Concatenate: original 210 values + memory delta 116 values = 326 total
    X_flat = np.concatenate([X_flat_raw, X_flat_delta], axis=1)   # (N, 326)
    case_windows[case]['X_flat'] = X_flat
    print(f'  {case}: raw {X_flat_raw.shape} + delta {X_flat_delta.shape} -> {X_flat.shape}')

print()
print(f'Input to PCA: {X_flat.shape[1]} dimensions  (210 original + 116 memory delta)')

Step 3a — Computing memory delta features ...

  single_case1: (59670, 30, 4) -> diff -> (59670, 29, 4)
  single_case2: (38043, 30, 4) -> diff -> (38043, 29, 4)
  complex_case1: (221495, 30, 4) -> diff -> (221495, 29, 4)
  complex_case2: (76950, 30, 4) -> diff -> (76950, 29, 4)

  Delta meaning: X_delta[window, t] = memory[t+1] - memory[t]
  Positive delta = memory growing  |  Negative = shrinking  |  ~0 = stable

Step 3b — Flattening and concatenating ...

  single_case1: raw (59670, 210) + delta (59670, 116) -> (59670, 326)
  single_case2: raw (38043, 210) + delta (38043, 116) -> (38043, 326)
  complex_case1: raw (221495, 210) + delta (221495, 116) -> (221495, 326)
  complex_case2: raw (76950, 210) + delta (76950, 116) -> (76950, 326)

Input to PCA: 326 dimensions  (210 original + 116 memory delta)


---
## Step 4 — PCA Transformation
- Fit PCA **only on SC1 normal windows** — avoids data leakage
- Keep components that explain 95% of variance (should be 28 components, same as original)
- Apply the same PCA to all cases

In [13]:
sc1_all      = case_windows['single_case1']
sc1_norm_msk = sc1_all['y'] == 0
X_pca_fit    = sc1_all['X_flat'][sc1_norm_msk]

print(f'Fitting PCA on {len(X_pca_fit):,} SC1 normal windows (210 dims) ...')

pca_full  = PCA()
pca_full.fit(X_pca_fit)
cumvar    = np.cumsum(pca_full.explained_variance_ratio_)
n_comp    = int(np.argmax(cumvar >= PCA_VARIANCE)) + 1
var_kept  = cumvar[n_comp - 1] * 100
print(f'Components for {PCA_VARIANCE*100:.0f}% variance: {n_comp}  (variance kept: {var_kept:.2f}%)')

pca = PCA(n_components=n_comp, random_state=RANDOM_SEED)
pca.fit(X_pca_fit)

print(f'\nApplying PCA (210 -> {n_comp}) to all cases ...')
for case in case_windows:
    X_pca = pca.transform(case_windows[case]['X_flat'])
    case_windows[case]['X_pca'] = X_pca
    print(f'  {case}: {case_windows[case]["X_flat"].shape} -> {X_pca.shape}')

pca_path = os.path.join(OUT_DIR, 'pca_model.pkl')
joblib.dump(pca, pca_path)
print(f'\nPCA model saved: {pca_path}')

Fitting PCA on 59,467 SC1 normal windows (210 dims) ...
Components for 95% variance: 31  (variance kept: 95.10%)

Applying PCA (210 -> 31) to all cases ...
  single_case1: (59670, 326) -> (59670, 31)
  single_case2: (38043, 326) -> (38043, 31)
  complex_case1: (221495, 326) -> (221495, 31)
  complex_case2: (76950, 326) -> (76950, 31)

PCA model saved: c:\Users\jthar\Documents\Claude\Projects\module3\data\processed\windows_v4\pca_model.pkl


---
## Step 5 — Option 4 Split

```
SC1 normals (all)      ──┐
SC2 normals (all)      ──┼──> X_train_pool -> 90% X_train + 10% X_val
CC1 normals (80%)      ──┘

CC1 normals (20%)  ──┐
CC1 anomalies      ──┤
SC1 anomalies      ──┼──> X_test_simple   (simple/sudden faults)
SC2 anomalies      ──┘

CC2 all            ──> X_test_drift        (complex/gradual faults)
```

**Key rules:**
- CC1 normal split is **temporal** (first 80% → train, last 20% → test) to avoid future data leakage
- SC1 and SC2 **anomaly** windows go to `X_test_simple` — not used in training, not wasted
- `X_val` is randomly carved from the training pool — used for early stopping only

In [14]:
sc1 = case_windows['single_case1']
sc2 = case_windows['single_case2']
cc1 = case_windows['complex_case1']
cc2 = case_windows['complex_case2']

# ── SC1: normals -> training, anomalies -> test_simple ───────────────
sc1_norm_msk = sc1['y'] == 0
sc1_anom_msk = sc1['y'] == 1
sc1_X_norm   = sc1['X_pca'][sc1_norm_msk]
sc1_X_anom   = sc1['X_pca'][sc1_anom_msk]
sc1_y_anom   = sc1['y'][sc1_anom_msk]
sc1_ft_anom  = sc1['ft'][sc1_anom_msk]

# ── SC2: normals -> training, anomalies -> test_simple ───────────────
sc2_norm_msk = sc2['y'] == 0
sc2_anom_msk = sc2['y'] == 1
sc2_X_norm   = sc2['X_pca'][sc2_norm_msk]
sc2_X_anom   = sc2['X_pca'][sc2_anom_msk]
sc2_y_anom   = sc2['y'][sc2_anom_msk]
sc2_ft_anom  = sc2['ft'][sc2_anom_msk]

# ── CC1: temporal 80/20 split on normals, all anomalies -> test_simple
cc1_norm_msk = cc1['y'] == 0
cc1_anom_msk = cc1['y'] == 1
cc1_X_norm   = cc1['X_pca'][cc1_norm_msk]
cc1_ft_norm  = cc1['ft'][cc1_norm_msk]
cc1_X_anom   = cc1['X_pca'][cc1_anom_msk]
cc1_y_anom   = cc1['y'][cc1_anom_msk]
cc1_ft_anom  = cc1['ft'][cc1_anom_msk]

cc1_split    = int(len(cc1_X_norm) * CC1_TRAIN_RATIO)
cc1_X_tr     = cc1_X_norm[:cc1_split]     # first 80% -> training
cc1_X_ts     = cc1_X_norm[cc1_split:]     # last  20% -> test_simple normals
cc1_ft_tr    = cc1_ft_norm[:cc1_split]
cc1_ft_ts    = cc1_ft_norm[cc1_split:]

print('Case breakdown:')
print(f'  SC1  normals -> train : {len(sc1_X_norm):>7,}  |  anomalies -> test_simple: {len(sc1_X_anom):,}')
print(f'  SC2  normals -> train : {len(sc2_X_norm):>7,}  |  anomalies -> test_simple: {len(sc2_X_anom):,}')
print(f'  CC1  normals -> train : {len(cc1_X_tr):>7,}  (80%)')
print(f'  CC1  normals -> test  : {len(cc1_X_ts):>7,}  (20%)  |  anomalies -> test_simple: {len(cc1_X_anom):,}')
print(f'  CC2  all     -> test_drift: {len(cc2["X_pca"]):,}')
print()

# ── Assemble training pool (normals only) ────────────────────────────
X_pool = np.concatenate([sc1_X_norm, sc2_X_norm, cc1_X_tr], axis=0)
y_pool = np.zeros(len(X_pool), dtype=np.int8)

print(f'Training pool total: {len(X_pool):,}  (SC1 {len(sc1_X_norm):,} + SC2 {len(sc2_X_norm):,} + CC1-80% {len(cc1_X_tr):,})')
print()

# ── Carve out validation set (10% of pool, random shuffle) ──────────
X_train, X_val, y_train, y_val = train_test_split(
    X_pool, y_pool,
    test_size=VAL_RATIO,
    random_state=RANDOM_SEED,
    shuffle=True
)
print(f'Train / Val ({int((1-VAL_RATIO)*100)}% / {int(VAL_RATIO*100)}%):')
print(f'  X_train : {X_train.shape}  <- VAE trains here')
print(f'  X_val   : {X_val.shape}  <- early stopping')
print()

# ── Test Simple: CC1 held-out normals + SC1/SC2/CC1 anomalies ────────
X_test_simple  = np.concatenate([
    cc1_X_ts,    # CC1 normals 20%
    sc1_X_anom,  # SC1 anomalies (cpu, memory, loss faults)
    sc2_X_anom,  # SC2 anomalies (pod-failure faults)
    cc1_X_anom,  # CC1 anomalies (5 fault types)
], axis=0)
y_test_simple  = np.concatenate([
    np.zeros(len(cc1_X_ts), dtype=np.int8),
    sc1_y_anom,
    sc2_y_anom,
    cc1_y_anom,
], axis=0)
ft_test_simple = np.concatenate([
    cc1_ft_ts, sc1_ft_anom, sc2_ft_anom, cc1_ft_anom
], axis=0)

print('Test Simple:')
print(f'  CC1 normals 20%   : {len(cc1_X_ts):>7,}')
print(f'  SC1 anomalies     : {len(sc1_X_anom):>7,}  (cpu, memory, loss)')
print(f'  SC2 anomalies     : {len(sc2_X_anom):>7,}  (pod-failure)')
print(f'  CC1 anomalies     : {len(cc1_X_anom):>7,}  (5 fault types)')
print(f'  X_test_simple     : {X_test_simple.shape}')
print(f'  Normals           : {(y_test_simple==0).sum():>7,}')
print(f'  Anomalies         : {(y_test_simple==1).sum():>7,}  ({y_test_simple.mean()*100:.2f}%)')
print()

# ── Test Drift: all CC2 ──────────────────────────────────────────────
X_test_drift  = cc2['X_pca']
y_test_drift  = cc2['y']
ft_test_drift = cc2['ft']

print('Test Drift (all CC2):')
print(f'  X_test_drift : {X_test_drift.shape}')
print(f'  Normals      : {(y_test_drift==0).sum():>7,}')
print(f'  Anomalies    : {(y_test_drift==1).sum():>7,}  ({y_test_drift.mean()*100:.2f}%)')

Case breakdown:
  SC1  normals -> train :  59,467  |  anomalies -> test_simple: 203
  SC2  normals -> train :  37,958  |  anomalies -> test_simple: 85
  CC1  normals -> train : 176,569  (80%)
  CC1  normals -> test  :  44,143  (20%)  |  anomalies -> test_simple: 783
  CC2  all     -> test_drift: 76,950

Training pool total: 273,994  (SC1 59,467 + SC2 37,958 + CC1-80% 176,569)

Train / Val (90% / 10%):
  X_train : (246594, 31)  <- VAE trains here
  X_val   : (27400, 31)  <- early stopping

Test Simple:
  CC1 normals 20%   :  44,143
  SC1 anomalies     :     203  (cpu, memory, loss)
  SC2 anomalies     :      85  (pod-failure)
  CC1 anomalies     :     783  (5 fault types)
  X_test_simple     : (45214, 31)
  Normals           :  44,143
  Anomalies         :   1,071  (2.37%)

Test Drift (all CC2):
  X_test_drift : (76950, 31)
  Normals      :  75,471
  Anomalies    :   1,479  (1.92%)


---
## Step 6 — Save All Outputs
All files go to `data/processed/windows_v4/` — completely separate from the original `windows/` directory.

In [15]:
saves = {
    'X_train'       : X_train,
    'y_train'       : y_train,
    'X_val'         : X_val,
    'y_val'         : y_val,
    'X_test_simple' : X_test_simple,
    'y_test_simple' : y_test_simple,
    'X_test_drift'  : X_test_drift,
    'y_test_drift'  : y_test_drift,
    'ft_test_simple': ft_test_simple,
    'ft_test_drift' : ft_test_drift,
}

print(f'Saving to: {OUT_DIR}\n')
for name, arr in saves.items():
    path    = os.path.join(OUT_DIR, f'{name}.npy')
    np.save(path, arr)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f'  {name:22s}: shape={str(arr.shape):20s}  {size_mb:.1f} MB')

print(f'\nPCA model : {pca_path}')

Saving to: c:\Users\jthar\Documents\Claude\Projects\module3\data\processed\windows_v4

  X_train               : shape=(246594, 31)          29.2 MB
  y_train               : shape=(246594,)             0.2 MB
  X_val                 : shape=(27400, 31)           3.2 MB
  y_val                 : shape=(27400,)              0.0 MB
  X_test_simple         : shape=(45214, 31)           5.3 MB
  y_test_simple         : shape=(45214,)              0.0 MB
  X_test_drift          : shape=(76950, 31)           9.1 MB
  y_test_drift          : shape=(76950,)              0.1 MB
  ft_test_simple        : shape=(45214,)              0.4 MB
  ft_test_drift         : shape=(76950,)              0.7 MB

PCA model : c:\Users\jthar\Documents\Claude\Projects\module3\data\processed\windows_v4\pca_model.pkl


---
## Step 7 — Verification

In [16]:
print('=' * 68)
print('OPTION 4 SPLIT — FINAL VERIFICATION')
print('=' * 68)
print()
print(f'  {"Split":<20} {"Shape":<22} {"Normals":>9} {"Anomalies":>10} {"Anom%":>7}')
print('-' * 68)

splits = [
    ('X_train',       X_train,       y_train),
    ('X_val',         X_val,         y_val),
    ('X_test_simple', X_test_simple, y_test_simple),
    ('X_test_drift',  X_test_drift,  y_test_drift),
]
for name, X, y in splits:
    n0  = (y == 0).sum()
    n1  = (y == 1).sum()
    pct = y.mean() * 100
    print(f'  {name:<20} {str(X.shape):<22} {n0:>9,} {n1:>10,} {pct:>6.2f}%')

print('-' * 68)
total_train = len(X_train) + len(X_val)
total_test  = len(X_test_simple) + len(X_test_drift)
ratio       = total_test / total_train

print()
print(f'  Training total (train + val)      : {total_train:>10,}')
print(f'  Testing  total (simple + drift)   : {total_test:>10,}')
print(f'  Train : Test ratio                :  1 : {ratio:.2f}')
print()

# NaN and range checks
print('  Integrity checks:')
all_pass = True
for name, X, _ in splits:
    nans   = int(np.isnan(X).sum())
    status = 'PASS' if nans == 0 else f'FAIL ({nans} NaNs)'
    print(f'    NaN check  {name:<20}: {status}')
    if nans > 0:
        all_pass = False

# Confirm training set contains no anomalies
train_anom = int(y_train.sum()) + int(y_val.sum())
status = 'PASS' if train_anom == 0 else f'FAIL ({train_anom} anomalies found!)'
print(f'    No anomalies in train/val       : {status}')
if train_anom > 0:
    all_pass = False

print()
print('=' * 68)
print(f'RESULT: {"ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED — review above"}')
print('=' * 68)
print()
print('To load in VAE training notebook:')
print("  X_train = np.load('data/processed/windows_v4/X_train.npy')")
print("  X_val   = np.load('data/processed/windows_v4/X_val.npy')")
print("  pca     = joblib.load('data/processed/windows_v4/pca_model.pkl')")

OPTION 4 SPLIT — FINAL VERIFICATION

  Split                Shape                    Normals  Anomalies   Anom%
--------------------------------------------------------------------
  X_train              (246594, 31)             246,594          0   0.00%
  X_val                (27400, 31)               27,400          0   0.00%
  X_test_simple        (45214, 31)               44,143      1,071   2.37%
  X_test_drift         (76950, 31)               75,471      1,479   1.92%
--------------------------------------------------------------------

  Training total (train + val)      :    273,994
  Testing  total (simple + drift)   :    122,164
  Train : Test ratio                :  1 : 0.45

  Integrity checks:
    NaN check  X_train             : PASS
    NaN check  X_val               : PASS
    NaN check  X_test_simple       : PASS
    NaN check  X_test_drift        : PASS
    No anomalies in train/val       : PASS

RESULT: ALL CHECKS PASSED

To load in VAE training notebook:
  X_train

---
## Output Files Summary

| File | Shape | Description |
|------|-------|-------------|
| `X_train.npy` | (N, ~30) | Normal windows — VAE trains here |
| `y_train.npy` | (N,) | All zeros |
| `X_val.npy` | (N, ~30) | Normal windows — early stopping |
| `y_val.npy` | (N,) | All zeros |
| `X_test_simple.npy` | (N, ~30) | CC1 held-out normals + SC1/SC2/CC1 anomalies |
| `y_test_simple.npy` | (N,) | Labels (0/1) |
| `X_test_drift.npy` | (N, ~30) | All CC2 windows |
| `y_test_drift.npy` | (N,) | Labels (0/1) |
| `ft_test_simple.npy` | (N,) | Fault types for test simple |
| `ft_test_drift.npy` | (N,) | Fault types for test drift |
| `pca_model.pkl` | — | Fitted PCA — reuse at inference time |

**Pipeline dimensions:**
```
Raw window:   (30, 7)   = 210 values
+ mem delta:  (29, 4)   = 116 values
Concat flat:             = 326 values  → input to PCA
PCA output:             ~30 components (95% variance of 326-dim input)
VAE input:              ~30 dimensions
```

**Load in VAE training notebook:**
```python
import numpy as np, joblib

X_train = np.load('data/processed/windows_v4/X_train.npy')
X_val   = np.load('data/processed/windows_v4/X_val.npy')
pca     = joblib.load('data/processed/windows_v4/pca_model.pkl')

# At inference — preprocess a new raw window (30, 7):
# 1. Compute memory delta
mem_delta = np.diff(new_window[:, 3:7], axis=0)          # (29, 4)
# 2. Concatenate and flatten
flat = np.concatenate([new_window.flatten(), mem_delta.flatten()])  # (326,)
# 3. PCA transform
pca_input = pca.transform(flat.reshape(1, -1))            # (1, n_components)
```